# 09 - How fragile are the allocation decisions?

Step 9.1 of `THESIS_GUIDE.md` (PIANO.md C4). Notebook 04 showed that two seeds of the same model share only part of their portfolio. This notebook asks why, by comparing two quantities on the same stock and day (`xaifin/portfolio/fragility.py`):

- the **margin** of a held stock to the top-k boundary, in standard deviations of the day's scores (`margin_z`, chapter 7): how far its score could fall before another stock overtakes it;
- the **seed noise** of the stock-day: the standard deviation, across the seeds of the same model, of the stock's daily z-scored score, i.e. how much training luck moves its position in the cross-section.

A held stock whose margin is smaller than its seed noise (ratio < 1) is held or not depending on the seed: the decision is fragile. If the measure is right, the share of such holdings should match the share of the portfolio that actually changes between two seeds.

Core grid, T = 20, L = 5, every seed available.

In [ ]:
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from xaifin import viz
from xaifin.config import CORE_UNIVERSES, RESULTS_ROOT, TOP_K, UNIVERSE_NAMES
from xaifin.portfolio.backtest import find_portfolios
from xaifin.portfolio.fragility import fragility, seed_noise, summary, zscores
from xaifin.portfolio.topk import overlap

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
SEQ_LEN, PRED_LEN = 20, 5
portfolios = find_portfolios()
portfolios = portfolios[(portfolios["seq_len"] == SEQ_LEN) & (portfolios["pred_len"] == PRED_LEN)
                        & portfolios["universe"].isin(CORE_UNIVERSES) & (portfolios["k"] == portfolios["universe"].map(TOP_K))]
portfolios = portfolios[portfolios["path"].map(lambda p: (p / "holdings.parquet").exists())]
print(len(portfolios), "portfolios with holdings")

## 1. Margins against seed noise

Per model and universe: the median seed noise and margin of the held stocks, the median ratio, the share of holdings closer to the boundary than their seed noise (and than half of it), and, for comparison, the turnover actually observed between the portfolios of two seeds (mean over pairs of seeds and rebalances).

In [ ]:
rows, held_all = [], []
for (model, universe), group in portfolios.groupby(["model", "universe"]):
    seeds = sorted(group["seed"])
    if len(seeds) < 2:
        continue
    noise = seed_noise(zscores(model, universe, SEQ_LEN, PRED_LEN, seeds=seeds))
    holdings = {r.seed: pd.read_parquet(r.path / "holdings.parquet") for r in group.itertuples()}
    held = pd.concat([fragility(h, noise).assign(seed=s) for s, h in holdings.items()])
    held_all.append(held.assign(model=model, universe=universe))
    observed = np.mean([overlap(holdings[a], holdings[b])["turnover"].mean() for a, b in combinations(seeds, 2)])
    rows.append({"model": model, "universe": universe, "seeds": len(seeds), **summary(held), "observed seed turnover": observed})
table = pd.DataFrame(rows)
table["model"] = pd.Categorical(table["model"], viz.MODEL_ORDER)
table = table.sort_values(["universe", "model"]).reset_index(drop=True)
table

## 2. Does the measure predict what happens?

Each point is one model in one universe: the share of its holdings with a margin below their seed noise against the share of the portfolio that changes from one seed to another. Points on the diagonal mean that the margins and the seed noise explain the instability of the portfolios between seeds.

In [ ]:
fig = px.scatter(table, x="share_below_noise", y="observed seed turnover", color="model", symbol="universe",
                 color_discrete_sequence=viz.THEMES[False]["series"], category_orders={"model": viz.MODEL_ORDER},
                 labels={"share_below_noise": "holdings with margin < seed noise", "observed seed turnover": "turnover between seeds"},
                 title="Fragile holdings against the portfolio change between seeds")
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(color=viz.THEMES[False]["muted"], dash="dot", width=1))
fig.update_xaxes(range=[0, 1], tickformat=".0%")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()
print("correlation:", round(table["share_below_noise"].corr(table["observed seed turnover"]), 3),
      "| mean absolute gap:", round((table["share_below_noise"] - table["observed seed turnover"]).abs().mean(), 3))

## 3. Distribution of the ratio margin / seed noise

All held stock-days of a model, pooled over universes and seeds; the dotted line at 1 separates the decisions that a new seed can overturn.

In [ ]:
held_all = pd.concat(held_all)
fig = px.histogram(held_all[held_all["ratio"] < 5], x="ratio", color="model", nbins=50, histnorm="probability",
                   facet_col="model", facet_col_wrap=3, category_orders={"model": viz.MODEL_ORDER},
                   color_discrete_sequence=viz.THEMES[False]["series"], title="Margin / seed noise of the held stocks (cut at 5)")
fig.add_vline(x=1, line_dash="dot", line_color=viz.THEMES[False]["muted"])
fig.update_layout(showlegend=False)
fig.show()
held_all.groupby("model")["ratio"].describe(percentiles=[0.1, 0.25, 0.5, 0.75]).reindex(viz.MODEL_ORDER)

## 4. Saved summary

`results/analysis/decision_fragility.csv`: the table of section 1.

In [ ]:
table.to_csv(RESULTS_ROOT / "analysis" / "decision_fragility.csv", index=False)
print("saved")

## Observations

*To be written once the three seeds of every run are available (PIANO.md C4). First reading on seeds 0 and 5: the share of holdings closer to the boundary than their seed noise matches the observed turnover between seeds within a few points (HIST DJI 0.44 vs 0.42, FinFormer DJI 0.67 vs 0.70).*